In [ ]:
!pip install netron


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 45.4 MB/s eta 0:00:00


In [ ]:
# Import required packages
import cv2
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten
from keras.optimizers import Adam
from keras.preprocessing.image import ImageDataGenerator

# Initialize image data generator with rescaling
train_data_gen = ImageDataGenerator(rescale=1./255) #preprocess all images inside the train dataset
validation_data_gen = ImageDataGenerator(rescale=1./255) #preprocess all images inside the validation dataset

# Preprocess all train images
train_generator = train_data_gen.flow_from_directory(
        '/content/drive/MyDrive/ColabNotebooks/dataset1/train',
        target_size=(48, 48),
        batch_size=64,
        color_mode="grayscale", #1 color channel
        class_mode='categorical')

# Preprocess all validation images
validation_generator = validation_data_gen.flow_from_directory(
        '/content/drive/MyDrive/ColabNotebooks/data/validation',
        target_size=(48, 48),
        batch_size=64,
        color_mode="grayscale",
        class_mode='categorical')

# Create CNN model
emotion_model = Sequential()

# Add convolutional layers with increasing filters and kernel sizes
emotion_model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(48, 48, 1)))
emotion_model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Dropout(0.25))

emotion_model.add(Conv2D(128, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Conv2D(256, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Dropout(0.25))

emotion_model.add(Conv2D(512, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Dropout(0.25))

# Flatten the output from the convolutional layers
emotion_model.add(Flatten())

# Add dense layers with dropout regularization
emotion_model.add(Dense(512, activation='relu'))
emotion_model.add(Dropout(0.5))
emotion_model.add(Dense(256, activation='relu'))
emotion_model.add(Dropout(0.5))
emotion_model.add(Dense(128, activation='relu'))
emotion_model.add(Dropout(0.5))
emotion_model.add(Dense(64, activation='relu'))
emotion_model.add(Dropout(0.5))

# Add the output layer with softmax activation for multiclass classification
emotion_model.add(Dense(7, activation='softmax'))

cv2.ocl.setUseOpenCL(False)

# Compile the model with categorical crossentropy loss and Adam optimizer
emotion_model.compile(loss='categorical_crossentropy', optimizer=Adam(lr=0.0001, decay=1e-6), metrics=['accuracy'])

# Train the model using the data generator with 50 epochs
history = emotion_model.fit_generator(train_generator,steps_per_epoch=train_generator.samples // train_generator.batch_size,epochs=50, validation_data=validation_generator, validation_steps=validation_generator.samples // validation_generator.batch_size)


# Plot the accuracy and loss
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

# save model structure in jason file
model_json = emotion_model.to_json()
with open("/content/drive/MyDrive/ColabNotebooks/model/emotion_model_latest.json", "w") as json_file:
    json_file.write(model_json)

# save trained model weight in .h5 file
emotion_model.save_weights('/content/drive/MyDrive/ColabNotebooks/model/emotion_model_latest.h5')


ImportError: cannot import name 'ImageDataGenerator' from 'keras.preprocessing.image' (/usr/local/lib/python3.12/dist-packages/keras/preprocessing/image/__init__.py)

In [ ]:
!pip install tensorflow

In [ ]:
!pip install tf2onnx


In [ ]:
!pip install netron


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import model_from_json
import tf2onnx
import netron

# Load model architecture from JSON file
with open('/content/drive/MyDrive/ColabNotebooks/model/emotion_model_new.json', 'r') as f:
    model_json = f.read()
model = model_from_json(model_json)

# Load model weights from H5 file
model.load_weights('/content/drive/MyDrive/ColabNotebooks/model/emotion_model_new.h5')

# Export model to ONNX format
onnx_model_path = 'model.onnx'
tf2onnx.convert.from_keras(model, output_path=onnx_model_path)

# Visualize ONNX model using Netron
netron.start(onnx_model_path)


In [ ]:
!pip install ngrok


In [ ]:
!pip install --upgrade ngrok


In [ ]:
from google.colab.output import eval_js
from urllib.request import urlopen
import ngrok

import subprocess

proc = subprocess.Popen(['python', '-m', 'http.server', '18270'])

# Get the public URL for the local server
public_url = ngrok.connect(port='18270')
print(f'Public URL: {public_url}')

# Open the public URL in a new browser tab
eval_js(f'window.open("{public_url}")')


In [ ]:
from keras.models import model_from_json
from keras.utils.vis_utils import plot_model

# Load the model architecture from JSON file
with open('/content/drive/MyDrive/ColabNotebooks/model/emotion_model_new.json', 'r') as f:
    model_architecture = f.read()

model = model_from_json(model_architecture)

# Load the model weights from H5 file
model.load_weights('/content/drive/MyDrive/ColabNotebooks/model/emotion_model_new.h5')

# Visualize the model architecture
plot_model(model, to_file='model.png', show_shapes=True)

# Display the model architecture image
from IPython.display import Image
Image(filename='model.png')


In [ ]:
!pip install graphviz


In [ ]:
from keras.models import model_from_json
from keras.utils.vis_utils import model_to_dot
from IPython.display import Image

# Load the model architecture from JSON file
with open('/content/drive/MyDrive/ColabNotebooks/model/emotion_model_new.json', 'r') as f:
    model_architecture = f.read()

model = model_from_json(model_architecture)

# Load the model weights from H5 file
model.load_weights('/content/drive/MyDrive/ColabNotebooks/model/emotion_model_new.h5')

# Generate a Graphviz representation of the model architecture
graphviz_str = model_to_dot(model, show_shapes=True).to_string()

# Display the Graphviz representation of the model architecture
Image(graphviz_str)


In [ ]:
!apt install graphviz
!pip install pydot


In [ ]:
!pip install caffe

In [ ]:
from keras.models import model_from_json
import caffe.draw
from google.colab import drive
import os
from keras.models import model_from_json
import pydot
from IPython.display import Image

# Load the model architecture from the JSON file
with open('/content/drive/MyDrive/ColabNotebooks/model/emotion_model_new.json', 'r') as f:
    model_json = f.read()
model = model_from_json(model_json)

# Visualize the model architecture using pydot and graphviz
dot = pydot.Dot()
for layer in model.layers:
    dot.add_node(pydot.Node(str(layer.name)))
    if isinstance(layer, keras.layers.convolutional.Conv2D):
        for i in range(layer.kernel_size[0]):
            for j in range(layer.kernel_size[1]):
                for input_dim in range(layer.input_shape[3]):
                    dot.add_node(pydot.Node('{}-{}-{}-{}'.format(layer.name, i, j, input_dim)))
                    dot.add_edge(pydot.Edge('{}-{}-{}-{}'.format(layer.name, i, j, input_dim), layer.name))
    else:
        for node in layer.inbound_nodes:
            for inbound_layer in node.inbound_layers:
                dot.add_edge(pydot.Edge(inbound_layer.name, layer.name))

# Display the model architecture
Image(dot.create_png())


In [ ]:
!pip install tensorflow keras pydot graphviz
!apt-get install graphviz


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import model_from_json
from tensorflow.keras.utils import plot_model
import pydot
import graphviz
from IPython.display import Image
import tensorflow as tf
from tensorflow.keras.models import model_from_json
import pydot
from IPython.display import Image

# Load the saved model architecture from the json file
json_file = open('/content/drive/MyDrive/ColabNotebooks/model/emotion_model_new.json', 'r')
loaded_model_json = json_file.read()
json_file.close()
model = model_from_json(loaded_model_json)

# Load the saved model weights from the h5 file
model.load_weights('/content/drive/MyDrive/ColabNotebooks/model/emotion_model_new.h5')

# Define a function to visualize the model architecture
def visualize_model(model):
    # Convert the Keras model to a graph
    dot_model = tf.keras.utils.model_to_dot(model, show_shapes=True)

    # Convert the graph to an image
    png_bytes = dot_model.create_png()
    with open('model.png', 'wb') as f:
        f.write(png_bytes)

    # Display the image
    return Image('model.png')

# Visualize the model architecture
visualize_model(model)


In [ ]:
!pip install pydot graphviz

import pydot
from tensorflow.keras.models import model_from_json
from keras.utils.vis_utils import plot_model

# load the model architecture from JSON file
with open('/content/drive/MyDrive/ColabNotebooks/model/emotion_model_new.json', 'r') as f:
    model_architecture = f.read()

# create the Keras model from the JSON architecture
model = model_from_json(model_architecture)

# load the model weights from HDF5 file
model.load_weights('/content/drive/MyDrive/ColabNotebooks/model/emotion_model_new.h5')

# plot the model architecture using pydot and graphviz
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True)


In [ ]:
!pip install visualkeras

In [ ]:
import visualkeras
from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import Dense, Flatten, Conv2D, Dropout, MaxPooling2D, InputLayer, ZeroPadding2D
from collections import defaultdict
import visualkeras
from PIL import ImageFont

# create VGG16
image_size = 224
# Now visualize the model!

color_map = defaultdict(dict)
color_map[Conv2D]['fill'] = 'orange'
color_map[ZeroPadding2D]['fill'] = 'gray'
color_map[Dropout]['fill'] = 'pink'
color_map[MaxPooling2D]['fill'] = 'red'
color_map[Dense]['fill'] = 'green'
color_map[Flatten]['fill'] = 'teal'

#font = ImageFont.truetype("arial.ttf", 32)

# create model structure
emotion_model = Sequential()

emotion_model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(48, 48, 1)))
emotion_model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Dropout(0.25))

emotion_model.add(Conv2D(128, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Conv2D(128, kernel_size=(3, 3), activation='relu'))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Dropout(0.25))

emotion_model.add(Flatten())
emotion_model.add(Dense(1024, activation='relu'))
emotion_model.add(Dropout(0.5))
emotion_model.add(Dense(7, activation='softmax'))


visualkeras.layered_view(emotion_model, to_file='vgg16.png', type_ignore=[visualkeras.SpacingDummyLayer])
visualkeras.layered_view(emotion_model, to_file='vgg16_legend.png', type_ignore=[visualkeras.SpacingDummyLayer],
                         legend=True)
visualkeras.layered_view(emotion_model, to_file='vgg16_spacing_layers.png', spacing=0)
visualkeras.layered_view(emotion_model, to_file='vgg16_type_ignore.png',
                         type_ignore=[ZeroPadding2D, visualkeras.SpacingDummyLayer])
visualkeras.layered_view(emotion_model, to_file='vgg16_color_map.png',
                         color_map=color_map, type_ignore=[visualkeras.SpacingDummyLayer])
visualkeras.layered_view(emotion_model, to_file='vgg16_flat.png',
                         draw_volume=False, type_ignore=[visualkeras.SpacingDummyLayer])
visualkeras.layered_view(emotion_model, to_file='vgg16_scaling.png',
                         scale_xy=1, scale_z=1, max_z=20, type_ignore=[visualkeras.SpacingDummyLayer])
